# Sprint 2 – Modeling

**Goal:** find the model that writes the most bills **fully right**, and measure it on the test set.

We try 4 methods (nicknames from our "Five Ways to Write a Bill" page):

| Method | Nickname | Idea |
|---|---|---|
| Baseline | The Lazy One | Same bill (the most common) for everyone. The score to beat. |
| A – k-Nearest Neighbors | The Copier | Copies the bill of the most similar customer. |
| B – Binary Relevance | The 731 Specialists | One small decision tree per bill item (yes/no). |
| C – Decision Tree (multi-output) | The Rule Tree | One tree that fills all 731 items at once. |

*Label Powerset (The Fixed Menu) is not tested: only 3.5% of the test bills exist in train.*

**How we measure:**
- **EMR** = % of customers with **all 731** boxes right (the professor's metric). **This decides.**
- **Boxes right** = % of single boxes right. Only to understand how close a model gets.

**How we choose (the "funnel"):** quick test on a small sample → best one on all the data → final exam on `test.csv` (once).

## 1. Load the clean data (from `sprint2-cleaning.ipynb`)

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("train_clean.zip", dtype="uint8")
print("Rows, columns:", df.shape)

Rows, columns: (62622, 1474)


In [2]:
# X = input (CRM options), Y = output (BIL items we must predict)
crm_cols = [c for c in df.columns if c.startswith("CRM")]
bil_cols = [c for c in df.columns if c.startswith("BIL")]
X = df[crm_cols]
Y = df[bil_cols]
print("CRM columns:", len(crm_cols), "| BIL columns:", len(bil_cols))

CRM columns: 743 | BIL columns: 731


## 2. Split: 80% to learn, 20% to validate

The model learns on 80%. We compare models on the other 20% (the "practice exam").
`solution.csv` is only used once, at the end (the "final exam").

In [3]:
from sklearn.model_selection import train_test_split

# random_state=42 -> same split every time we run it
X_train, X_val, Y_train, Y_val = train_test_split(X, Y, test_size=0.2, random_state=42)
print("Train rows:", len(X_train), "| Validation rows:", len(X_val))

Train rows: 50097 | Validation rows: 12525


## 3. How we score a model

In [4]:
def score(name, predictions):
    # EMR: the whole row (all 731 boxes) must match
    pred = np.asarray(predictions)
    real = Y_val.values
    emr = (pred == real).all(axis=1).mean() * 100
    boxes = (pred == real).mean() * 100
    print(f"{name}: EMR = {emr:.2f}%  |  boxes right = {boxes:.2f}%")
    return emr

## 4. Quick test on a small sample (10,000 rows)

A small sample is enough to spot bad methods, and it's fast.

In [5]:
# take 10,000 random rows from the training part
X_small = X_train.sample(10000, random_state=42)
Y_small = Y_train.loc[X_small.index]
results = {}

### Baseline – The Lazy One

In [6]:
# the most common bill in the sample, given to every customer
most_common_bill = Y_small.value_counts().index[0]
pred = np.tile(np.array(most_common_bill), (len(X_val), 1))
results["Baseline"] = score("Baseline", pred)

Baseline: EMR = 0.02%  |  boxes right = 97.65%


### A – The Copier (k-Nearest Neighbors)

In [7]:
from sklearn.neighbors import KNeighborsClassifier

# n_neighbors=1 -> copy the bill of the single most similar customer
knn = KNeighborsClassifier(n_neighbors=1)
knn.fit(X_small, Y_small)
results["A - kNN"] = score("A - kNN", knn.predict(X_val))

A - kNN: EMR = 0.22%  |  boxes right = 99.43%


### B – The 731 Specialists (Binary Relevance)

`MultiOutputClassifier` trains one decision tree per bill item.
Trick: we give the data as a **sparse matrix** (stores only the 1s). Same result, about 10× faster, because most values are 0.

In [8]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.multioutput import MultiOutputClassifier
from scipy.sparse import csr_matrix

# n_jobs=-1 -> use all the computer's cores
specialists = MultiOutputClassifier(DecisionTreeClassifier(random_state=42), n_jobs=-1)
specialists.fit(csr_matrix(X_small.values), Y_small)
results["B - Specialists"] = score("B - Specialists", specialists.predict(csr_matrix(X_val.values)))

B - Specialists: EMR = 71.45%  |  boxes right = 99.61%


### C – The Rule Tree (one multi-output decision tree)

In [9]:
# one single tree that predicts all 731 items at once
tree = DecisionTreeClassifier(random_state=42)
tree.fit(X_small, Y_small)
results["C - Rule Tree"] = score("C - Rule Tree", tree.predict(X_val))

C - Rule Tree: EMR = 0.25%  |  boxes right = 99.29%


> **Why no Random Forest?** We tried a multi-output Random Forest (50 trees): it ran out of memory.
> Every leaf of every tree stores an answer for all 731 items, which is too much for a laptop.

### Compare

In [10]:
pd.Series(results).sort_values(ascending=False).round(2)

B - Specialists    71.45
C - Rule Tree       0.25
A - kNN             0.22
Baseline            0.02
dtype: float64

**How to read it:** the best method has the highest EMR, and it must be clearly above the Baseline.
In our first test, **B (The 731 Specialists)** won by far. Why: each item gets its own simple rule
("if the customer has option X, bill item Y"). The single tree (C) mixes all 731 items in one set of rules and can't specialize,
and the Copier (A) copies the small differences of the neighbor.

## 5. The winner on all the training data (80%)

Now we train the winner on the full 80% and check it again on the 20%.

In [11]:
winner = MultiOutputClassifier(DecisionTreeClassifier(random_state=42), n_jobs=-1)
winner.fit(csr_matrix(X_train.values), Y_train)
score("B - Specialists (all data)", winner.predict(csr_matrix(X_val.values)))

B - Specialists (all data): EMR = 73.27%  |  boxes right = 99.46%


np.float64(73.26946107784431)

## 6. Final exam: predict `test.csv` and compare with `solution.csv` (only once)

We train the winner on **100%** of the clean data, then predict the test customers.

⚠️ **Reminder:** the test set must have the **same columns** as train. Using `crm_cols` removes the 2 constant columns (`CRM_2396_PACK`, `CRM_2398_PACK`) automatically.

In [12]:
final_model = MultiOutputClassifier(DecisionTreeClassifier(random_state=42), n_jobs=-1)
final_model.fit(csr_matrix(X.values), Y)

,estimator estimator: estimator objectAn estimator object implementing :term:`fit` and :term:`predict`.A :term:`predict_proba` method will be exposed only if `estimator` implementsit.,DecisionTreeC...ndom_state=42)
,"n_jobs n_jobs: int or None, optional (default=None)The number of jobs to run in parallel.:meth:`fit`, :meth:`predict` and :meth:`partial_fit` (if supportedby the passed estimator) will be parallelized for each target.When individual estimators are fast to train or predict,using ``n_jobs > 1`` can result in slower performance dueto the parallelism overhead.``None`` means `1` unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all available processes / threads.See :term:`Glossary ` for more details... versionchanged:: 0.20 `n_jobs` default changed from `1` to `None`.",-1
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.",'gini'
,"splitter splitter: {""best"", ""random""}, default=""best""The strategy used to choose the split at each node. Supportedstrategies are ""best"" to choose the best split and ""random"" to choosethe best random split.",'best'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: int, float or {""sqrt"", ""log2""}, default=NoneThe number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... note:: The search for a split does not stop until at least one valid partition of the node samples is found, even if it requires to effectively inspect more than ``max_features`` features.",None
,"random_state random_state: int, RandomState instance or None, default=NoneControls the randomness of the estimator. The features are alwaysrandomly permuted at each split, even if ``splitter`` is set to``""best""``. When ``max_features < n_features``, the algorithm willselect ``max_features`` at random at each split before finding the bestsplit among them. But the best found split may vary across differentruns, even if ``max_features=n_features``. That is the case, if theimprovement of the criterion is identical for 

In [13]:
test = pd.read_csv("test.csv")

# keep only the same 743 CRM columns used in training
dropped = [c for c in test.columns if c.startswith("CRM") and c not in crm_cols]
print("Columns dropped from test:", dropped)
X_test = test[crm_cols]

Columns dropped from test: ['CRM_2396_PACK', 'CRM_2398_PACK']


In [14]:
pred_test = final_model.predict(csr_matrix(X_test.values))

# turn each row of 731 zeros/ones into one text, like in solution.csv
bill_text = ["".join(row.astype(str)) for row in pred_test]

In [15]:
solution = pd.read_csv("solution.csv", dtype=str)
print("Same customers, same order:", (solution["MSISDN"].values == test["MSISDN"].values).all())

final_emr = (np.array(bill_text) == solution["Bill_Conf"].values).mean() * 100
print(f"FINAL EMR on test: {final_emr:.2f}%")

Same customers, same order: True
FINAL EMR on test: 94.46%


In [16]:
# save our predictions in the same format as solution.csv (zip: small enough for GitHub)
submission = pd.DataFrame({"MSISDN": test["MSISDN"], "Bill_Conf": bill_text})
submission.to_csv("predictions.zip", index=False, compression="zip")
print("Saved: predictions.zip")

Saved: predictions.zip


## 7. Next step for the team: MCA (with vs. without)

MCA "summarizes" the 743 CRM options into fewer columns **before** the model.
To test it fairly:
1. Fit MCA **only on `X_train`**, then apply the same MCA to `X_val` (and later to `X_test`).
2. Train method B on the MCA columns and use `score(...)` on the validation set.
3. Compare with B without MCA (section 5). **Use MCA only if the EMR is equal or higher.**
4. Only then repeat the final exam (section 6) with the chosen version.

Risk to watch: MCA keeps the big patterns and can lose rare options, and many bill items are rare.